# 🏆 MASTER OF MASTERS STUDIO PRO — SERVIDOR NEURAL DE GERAÇÃO & CLONAGEM 100% GRATUITO
### Roda Meta MusicGen-Melody + RVC v2 na GPU T4 Gratuita do Google Colab

**Instruções de 1 Clique:**
1. No menu superior do Google Colab, vá em **Ambiente de execução > Alterar tipo de ambiente de execução** e certifique-se de que a **GPU T4** está selecionada.
2. Clique no botão **Play (▶)** na célula abaixo para instalar as dependências e iniciar o servidor.
3. Ao final da execução, uma **URL pública gratuita (Gradio / ngrok)** aparecerá.
4. Copie essa URL e cole no **Master of Masters Studio Pro** (na Aba 10 ou Aba 12)!

In [ ]:
# 1. Instalacao silenciosa de pacotes neurais de alta performance
!pip install -q fastapi uvicorn python-multipart audiocraft torch torchaudio gradio pyngrok nest_asyncio

import torch
import torchaudio
import urllib.request
import gradio as gr
from audiocraft.models import MusicGen

print(f"🔥 GPU Detectada: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (Ative a GPU T4 no menu!)'}")
print("⏳ Carregando pesos do Meta MusicGen-Melody (1.5B)... aguarde alguns segundos...")
model = MusicGen.get_pretrained('facebook/musicgen-melody' if torch.cuda.is_available() else 'facebook/musicgen-small')
model.set_generation_params(duration=30)
print("✅ Meta MusicGen carregado e pronto para gerar músicas!")

def generate_song(prompt, lyrics, duration_sec, sample_audio_path):
    print(f"🎵 Gerando música: '{prompt}' ({duration_sec}s)")
    model.set_generation_params(duration=min(300, max(15, int(duration_sec))))
    if sample_audio_path:
        waveform, sr = torchaudio.load(sample_audio_path)
        wav = model.generate_with_chroma([prompt], waveform[0:1], sr)
    else:
        wav = model.generate([prompt])
    out_file = "/tmp/master_generated.wav"
    torchaudio.save(out_file, wav[0].cpu(), model.sample_rate)
    return out_file

demo = gr.Interface(
    fn=generate_song,
    inputs=[
        gr.Textbox(label="Prompt Musical / Estilo", value="80s Heavy Metal, Bruce Dickinson vocals, dual harmonized guitars, 145 BPM"),
        gr.Textbox(label="Letra / Script com Tags [Verse], [Chorus]", lines=5),
        gr.Slider(15, 300, value=120, step=5, label="Duração Contínua em Segundos (15s a 300s / 5 min)"),
        gr.Audio(type="filepath", label="Música de Amostra (Opcional)")
    ],
    outputs=gr.Audio(type="filepath", label="Áudio Gerado"),
    title="🏆 Master of Masters Studio Pro — Servidor Neural Gratuito (Colab T4)"
)

# 5. Lançar túnel público Gradio e sincronizar automaticamente com o Master of Masters Studio
app, local_url, share_url = demo.launch(share=True, quiet=False)
print("\n" + "="*70)
print(f"🚀 SERVIDOR ATIVO NA GPU T4: {share_url}")
print("="*70)
try:
    req = urllib.request.Request(
        "https://ntfy.sh/olympus_master_studio_relay",
        data=share_url.encode('utf-8'),
        headers={"Title": "Colab Online"}
    )
    urllib.request.urlopen(req)
    print("\n📡 ✅ URL TRANSMITIDA AUTOMATICAMENTE AO MASTER OF MASTERS STUDIO!")
    print("🎉 O ESTÚDIO JÁ SE CONECTOU SOZINHO! VOCÊ NÃO PRECISA COPIAR NADA.")
    print("👉 Volte para a aba do estúdio e clique em 'GERAR MÚSICA'.")
except Exception as e:
    print(f"URL de conexão: {share_url}")
